# Projeto 4: Classificação binária brest cancer - classificar somente um registro e salvar o classificador


## Etapa 1: Importação das bibliotecas

ambiente anaconda YouTube


In [1]:
# usar o melhor parametro 
#importação das bibliotecas

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score
import torch
import torch.nn as nn
torch.__version__

'2.8.0+cu128'

## Etapa 2: Base de dados

In [3]:
np.random.seed(123) # colocar sempre no msmo valor o random
torch.manual_seed(123)

In [4]:
previsores = pd.read_csv('entradas_breast.csv')
classe = pd.read_csv('saidas_breast.csv')

In [5]:
coluna_previsores = previsores.columns
coluna_classe = classe.columns
coluna_previsores, coluna_classe

(Index([' radius_mean', ' texture_mean', ' perimeter_mean', ' area_mean',
        ' smoothness_mean', ' compactness_mean', ' concavity_mean',
        'concave_points_mean', ' symmetry_mean', ' fractal_dimension_mean',
        ' radius_se', ' texture_se', ' perimeter_se', ' area_se',
        ' smoothness_se', ' compactness_se', ' concavity_se',
        ' concave_points_se', ' symmetry_se', ' fractal_dimension_se',
        ' radius_worst', ' texture_worst', ' perimeter_worst', ' area_worst',
        ' smoothness_worst', ' compactness_worst', ' concavity_worst',
        ' concave_points_worst', ' symmetry_worst', ' fractal_dimension_worst'],
       dtype='object'),
 Index(['0'], dtype='object'))

In [6]:
previsores = torch.tensor(np.array(previsores), dtype = torch.float)
classe = torch.tensor(np.array(classe), dtype = torch.float)

In [7]:
# Transformamos no tensos, como não usamos GridSearchCV e validação cruzada não precisamos fazer a interface com o sklearn

In [8]:
type(previsores)

torch.Tensor

## Etapa 3: Transformação dos dados para tensores


In [9]:
# concatenamos dados que são previsores e classe
# batch_size = 10
train_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(previsores, classe), batch_size = 30, shuffle = True)

## Etapa 4: Construção do modelo

Resposta 

{   'batch_size': 30,
    'criterion': <class 'torch.nn.modules.loss.BCEWithLogitsLoss'>,
    'max_epochs': 100,
    'module__activation': <function relu at 0x76f9e8532700>,
    'module__initializer': <function _make_deprecate.<locals>.deprecated_init at 0x76f9e85c47c0>,
    'module__neurons': 16,
    'optimizer': <class 'torch.optim.adam.Adam'>}

In [10]:
class classificador_torch(nn.Module):
    def __init__(self):
        super().__init__()

        # processo 30 neurônios na primeira camada  de entrada e 16 neurônios na primeira camada oculta
        self.dense0 = nn.Linear(30, 16)
        # Mesmos parâmetors usados pelo kernel initializer do Keras [https://keras.io/initializers/, ver sessão RandomNormal]
        torch.nn.init.normal_(self.dense0.weight, mean = 0.0, std= 0.05) # mesma configuração no keras, valor da média e o desvio padrão
        # camadas ocultas, duas, cada uma com 16 neurônios
        self.dense1 = nn.Linear(16, 16)
        torch.nn.init.normal_(self.dense1.weight, mean = 0.0, std= 0.05)
        # segunda camada oculta com 16 neurônios e 1 neurônio na camada de saída. 
        self.dense2 = nn.Linear(16, 1)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
        self.output = nn.Sigmoid()

    # foward faz ligação em todas as camadas. 
    def forward(self, X):
        X = self.dense0(X)
        X = self.activation(X)
        X = self.dropout(X)
        X = self.dense1(X)
        X = self.activation(X)
        X = self.dropout(X)
        X = self.dense2(X)
        X = self.output(X)
        return X

In [11]:
classificador = classificador_torch() # classificador que recebe a class module do pytorch

In [12]:
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(classificador.parameters(), lr = 0.001,
                             weight_decay = 0.0001)

## Etapa 5: Treinamento do modelo

In [13]:
# número de época escolhida foi de 100, epoch
for epoch in range(100):
    running_loss = 0.

    for data in train_loader:
        inputs, labels = data
        # a cada rodada é zerada o gradiente
        optimizer.zero_grad()
        
        # fazendo a classificação usando os inputs, previsores
        outputs = classificador(inputs)
        # cálculo do erro comparando as previsoes (outputs, com os dados reais labels)
        loss = criterion(outputs, labels)
        # processo do backward do neurônio é inicializado
        loss.backward()
        # atualização dos pesos => step
        optimizer.step()

        # somatória dos erros
        running_loss += loss.item()

    print('Época %3d: perda %.5f' % (epoch+1, running_loss/len(train_loader)))  # objetivo diminuir os erros

Época   1: perda 2.19596
Época   2: perda 1.05534
Época   3: perda 0.65136
Época   4: perda 0.54512
Época   5: perda 0.56162
Época   6: perda 0.52257
Época   7: perda 0.53002
Época   8: perda 0.52000
Época   9: perda 0.48141
Época  10: perda 0.47557
Época  11: perda 0.47348
Época  12: perda 0.44309
Época  13: perda 0.46689
Época  14: perda 0.46815
Época  15: perda 0.41412
Época  16: perda 0.41440
Época  17: perda 0.40691
Época  18: perda 0.39671
Época  19: perda 0.39836
Época  20: perda 0.39044
Época  21: perda 0.37104
Época  22: perda 0.36183
Época  23: perda 0.38659
Época  24: perda 0.36362
Época  25: perda 0.37057
Época  26: perda 0.37556
Época  27: perda 0.35010
Época  28: perda 0.33387
Época  29: perda 0.33808
Época  30: perda 0.30795
Época  31: perda 0.31508
Época  32: perda 0.33083
Época  33: perda 0.29238
Época  34: perda 0.32981
Época  35: perda 0.29265
Época  36: perda 0.31159
Época  37: perda 0.27537
Época  38: perda 0.30584
Época  39: perda 0.30039
Época  40: perda 0.27072


#### classificação do novo registro:

colocado manualmente

## Etapa 6: Classificar somente um registro

In [14]:
# são 30 atributos que vale a 30 entradas

novo = torch.tensor([[15.80, 8.34, 118, 900, 0.10, 0.26, 0.08, 0.134, 0.178,
                  0.20, 0.05, 1098, 0.87, 4500, 145.2, 0.005, 0.04, 0.05, 0.015,
                  0.03, 0.007, 23.15, 16.64, 178.5, 2018, 0.14, 0.185,
                  0.84, 158, 0.363]], dtype = torch.float)

In [15]:
coluna_previsores, coluna_classe

(Index([' radius_mean', ' texture_mean', ' perimeter_mean', ' area_mean',
        ' smoothness_mean', ' compactness_mean', ' concavity_mean',
        'concave_points_mean', ' symmetry_mean', ' fractal_dimension_mean',
        ' radius_se', ' texture_se', ' perimeter_se', ' area_se',
        ' smoothness_se', ' compactness_se', ' concavity_se',
        ' concave_points_se', ' symmetry_se', ' fractal_dimension_se',
        ' radius_worst', ' texture_worst', ' perimeter_worst', ' area_worst',
        ' smoothness_worst', ' compactness_worst', ' concavity_worst',
        ' concave_points_worst', ' symmetry_worst', ' fractal_dimension_worst'],
       dtype='object'),
 Index(['0'], dtype='object'))

In [16]:
len(coluna_previsores) 

30

In [17]:
len([15.80, 8.34, 118, 900, 0.10, 0.26, 0.08, 0.134, 0.178,
                  0.20, 0.05, 1098, 0.87, 4500, 145.2, 0.005, 0.04, 0.05, 0.015,
                  0.03, 0.007, 23.15, 16.64, 178.5, 2018, 0.14, 0.185,
                  0.84, 158, 0.363])

30

In [18]:
len(novo)

1

In [19]:
# classificar na forma de avaliação
classificador.eval()

classificador_torch(
  (dense0): Linear(in_features=30, out_features=16, bias=True)
  (dense1): Linear(in_features=16, out_features=16, bias=True)
  (dense2): Linear(in_features=16, out_features=1, bias=True)
  (activation): ReLU()
  (dropout): Dropout(p=0.2, inplace=False)
  (output): Sigmoid()
)

In [20]:
# atualização dos pesos, dropout, desligar alguns neurônios

In [21]:
# classificador com um novo registro
previsao = classificador(novo)

In [22]:
#valor de um tensor, e vamos buscar os dados aqui dentro
previsao

tensor([[1.]], grad_fn=<SigmoidBackward0>)

In [23]:
previsao = previsao.detach() #usando o comando detach para desvendar valor dentro da resposta previsao

In [24]:
previsao # acabamos pegando apenas a parte do tensor, que interessa

tensor([[1.]])

In [25]:
previsao = previsao.numpy()

In [26]:
previsao# transformar em numpy

array([[1.]], dtype=float32)

In [27]:
type(previsao)

numpy.ndarray

In [28]:
previsao # retornou uma probabilidade de 1.0, que quer dizer que é um câncer maligno

array([[1.]], dtype=float32)

In [29]:
# ver se é maior ou menor de 0,5, 50%

In [30]:
previsao = (previsao > 0.5)
previsao

array([[ True]])

Ficou True, então a probabilidade de ser um câncer maligno é verdadeiro. 

acima não mostramos a classe, e sim os dados de entrada previsores, para verificar se é 0 ou não 1 Câncer

# Etapa 7: Salvar o classificador

In [31]:
# Quando salvar, você precisa chamar classificador.state_dict() (com os parêntese no final),
# ao invés de classificador.state_dict

In [32]:
# mostrar todos os pesos
classificador.state_dict()

OrderedDict([('dense0.weight',
              tensor([[ 3.0197e-02,  1.8140e-01,  1.6734e-01, -3.2692e-04, -1.6838e-01,
                        4.5863e-04, -2.0909e-01,  6.0946e-02,  2.3099e-01,  2.0604e-01,
                        4.8386e-02, -9.7833e-02, -1.8874e-02, -7.5993e-02, -5.5547e-03,
                       -5.2949e-02,  8.0095e-02,  2.0435e-01, -3.8628e-03, -1.0071e-02,
                        2.1744e-02,  1.6817e-01,  2.7692e-02, -3.8436e-02, -8.1606e-02,
                        2.3104e-02, -4.2568e-02, -7.5688e-02,  1.3744e-01,  4.5682e-04],
                      [-4.9082e-02, -1.1097e-01, -2.0753e-01, -6.7641e-02,  1.6660e-01,
                        9.6760e-02,  9.3890e-02,  2.0316e-02, -6.9786e-02, -1.1045e-01,
                        3.5867e-03,  8.0619e-04,  6.4567e-03, -1.8769e-02,  1.1516e-02,
                        2.6228e-02, -4.3028e-02,  7.3910e-02,  5.5967e-02,  1.8892e-01,
                        1.0440e-02, -2.2173e-02, -1.9161e-01,  1.0880e-01, -4.7178e-02,


In [33]:
# salvar o arquivo
# Para salvar o classificador com a versão 1.5.0 do PyTorch, use o código abaixo

torch.save(classificador.state_dict(), 'checkpoint.pth')